In [1]:
import yfinance as yf
import pandas as pd 
import numpy as np

In [2]:
# SK하이닉스의 데이터를 수집 
hynix = yf.Ticker("000660.KS")
# 최근 2년간의 데이터를 추출 
df = hynix.history(period='2y')
df.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2024-05-20 00:00:00+09:00,189403.282097,190289.724915,185759.017178,187236.421875,3592128,0.0,0.0
2024-05-21 00:00:00+09:00,190486.739372,190585.233032,188319.878841,189107.828125,2468109,0.0,0.0
2024-05-22 00:00:00+09:00,189009.321217,194721.953125,188024.384682,194721.953125,3949439,0.0,0.0
2024-05-23 00:00:00+09:00,200434.590469,200927.058750,195214.426688,196987.312500,4681849,0.0,0.0
2024-05-24 00:00:00+09:00,196790.314435,199449.643009,194524.960465,195608.390625,3793872,0.0,0.0


In [3]:
# 종가, 거래량 데이터만 사용 
df = df[ ['Close', 'Volume'] ]

In [4]:
# 기술적 지표 : 20일간의 이동평균선 
# 20개의 데이터를 묶어준다. -> rolling(n)
df['MA20'] = df['Close'].rolling(20).mean()
df.iloc[ 15 : 25,  ]

,Close,Volume,MA20
Date,,,
2024-06-11 00:00:00+09:00,209299.015625,3070163,NaN
2024-06-12 00:00:00+09:00,211761.359375,2151331,NaN
2024-06-13 00:00:00+09:00,218655.890625,5777279,NaN
2024-06-14 00:00:00+09:00,217670.968750,3311223,NaN
2024-06-17 00:00:00+09:00,219640.843750,2198340,199942.117188
2024-06-18 00:00:00+09:00,230967.593750,3136231,202128.675781
2024-06-19 00:00:00+09:00,229982.687500,3758652,204172.418750
2024-06-20 00:00:00+09:00,233922.421875,2927358,206132.442188
2024-06-21 00:00:00+09:00,230475.171875,3848394,207806.835156


In [5]:
# 타켓 변수 -> 내일의 종가가 내일의 20일 평균선보다 높은가? (1 : 상승돌파, 0 : 하락)
# 인덱스를 한칸씩 위로 올린다. shift(1) -> 1칸씩 내린다 , shift(-1) -> -1칸씩 이동(위로 1칸 이동)
df['target'] = ( df['Close'].shift(-1) > df['MA20'].shift(-1) ).astype(int)

- astype() -> 타입을 변경하는 함수 (문자형 데이터를 숫자형으로 변경할때는 int)
    - 컬럼 내의 데이터가 '1' -> 1
    - '-' -> error
- to_numeric() -> 범주형 데이터를 수치형을 변경(문자로 되어있는 숫자를 숫자 타입으로 변경 )
    - '1' -> 1
    - '-' -> NaN

In [6]:
df['target'].value_counts()

target
1    295
0    190
Name: count, dtype: int64

In [7]:
# NLP 감성 점수 데이터를 추가 
# 원래는 기사를 크롤링하여 감성 점수를 예측하고 입력해야하지만 
# 무작위한 데이터를 입력 
df['NLP_Sentiment'] = np.random.uniform(-1, 1, len(df))

In [8]:
df['NLP_Sentiment'].describe()

count    485.000000
mean       0.036152
std        0.578527
min       -0.995635
25%       -0.484679
50%        0.046371
75%        0.518543
max        0.999358
Name: NLP_Sentiment, dtype: float64

In [9]:
# !pip install OpenDartReader

In [10]:
# 영업이익 데이터를 추가 
from opendartreader import OpenDartReader
from datetime import datetime
import os
from dotenv import load_dotenv

In [11]:
# .env 파일에서 환경 변수 로드 
load_dotenv()

True

In [12]:
api_key = os.getenv('api_key')

In [13]:
df.head(1)

,Close,Volume,MA20,target,NLP_Sentiment
Date,,,,,
2024-05-20 00:00:00+09:00,187236.421875,3592128,NaN,0,-0.242527


In [14]:
# 시차 정보를 제거 -> 시차 정보가 없는 시계열 데이터와 시차 정보가 존재하는 시계열 데이터가 결합 x
df.index = df.index.tz_localize(None)
df.head(1)


,Close,Volume,MA20,target,NLP_Sentiment
Date,,,,,
2024-05-20,187236.421875,3592128,NaN,0,-0.242527


In [15]:
# dart에서 데이터를 수집 
dart = OpenDartReader(api_key)

In [16]:
# 현재년도 로드 -> 최근 3년간 목록을 생성 
current_year = datetime.now().year
years = [ current_year - 2, current_year - 1, current_year ]
years

[2024, 2025, 2026]

In [17]:
# 분기 보고서 codes
reprt_codes = ['11013', '11012', '11014', '11011']
dart_data_list = []

In [18]:
for year in years:
    for code in reprt_codes:
        try : 
            report = dart.finstate('000660', year, code)
            if report is not None:
                op_profit = report[ (report['fs_div'] == 'CFS')  & ( report['account_nm'] == '영업이익' ) ]
                if not op_profit.empty:
                    val = int(
                            op_profit['thstrm_amount'].values[0].replace(',', '')
                        )
                    # code 11013 -> 1분기 보고서 
                    if code == '11013' : d = f"{year}-05-15"
                    elif code == '11012' : d = f"{year}-08-14"
                    elif code == '11014' : d = f"{year}-11-14"
                    else : d = f"{year+1}-03-31"

                    report_date = pd.to_datetime(d)
                    if report_date <= datetime.now():
                        dart_data_list.append({'Date' : report_date, 'Operation_Profit' : val})
        except: continue
dart_data_list

{'status': '013', 'message': '조회된 데이타가 없습니다.'}

{'status': '013', 'message': '조회된 데이타가 없습니다.'}

{'status': '013', 'message': '조회된 데이타가 없습니다.'}



[{'Date': Timestamp('2024-05-15 00:00:00'), 'Operation_Profit': 2886029000000},
 {'Date': Timestamp('2024-08-14 00:00:00'), 'Operation_Profit': 5468536000000},
 {'Date': Timestamp('2024-11-14 00:00:00'), 'Operation_Profit': 7029958000000},
 {'Date': Timestamp('2025-03-31 00:00:00'),
  'Operation_Profit': 23467319000000},
 {'Date': Timestamp('2025-05-15 00:00:00'), 'Operation_Profit': 7440504000000},
 {'Date': Timestamp('2025-08-14 00:00:00'), 'Operation_Profit': 9212851000000},
 {'Date': Timestamp('2025-11-14 00:00:00'),
  'Operation_Profit': 11383390000000},
 {'Date': Timestamp('2026-03-31 00:00:00'),
  'Operation_Profit': 47206319000000},
 {'Date': Timestamp('2026-05-15 00:00:00'),
  'Operation_Profit': 37610283000000}]

In [19]:
dart_df = pd.DataFrame(dart_data_list)
dart_df

,Date,Operation_Profit
0,2024-05-15,2886029000000
1,2024-08-14,5468536000000
2,2024-11-14,7029958000000
3,2025-03-31,23467319000000
4,2025-05-15,7440504000000
5,2025-08-14,9212851000000
6,2025-11-14,11383390000000
7,2026-03-31,47206319000000
8,2026-05-15,37610283000000


In [20]:
df.head()

,Close,Volume,MA20,target,NLP_Sentiment
Date,,,,,
2024-05-20,187236.421875,3592128,NaN,0,-0.242527
2024-05-21,189107.828125,2468109,NaN,0,0.747942
2024-05-22,194721.953125,3949439,NaN,0,0.926764
2024-05-23,196987.312500,4681849,NaN,0,-0.932733
2024-05-24,195608.390625,3793872,NaN,0,0.806539


In [ ]:
# df를 리셋 인덱스를 하고 결합 
df2 = df.reset_index()
pd.merge(df2, dart_df, on='Date', how='left').ffill()

,Date,Close,Volume,MA20,target,NLP_Sentiment,Operation_Profit
0,2024-05-20,1.872364e+05,3592128,NaN,0,-0.242527,NaN
1,2024-05-21,1.891078e+05,2468109,NaN,0,0.747942,NaN
2,2024-05-22,1.947220e+05,3949439,NaN,0,0.926764,NaN
3,2024-05-23,1.969873e+05,4681849,NaN,0,-0.932733,NaN
4,2024-05-24,1.956084e+05,3793872,NaN,0,0.806539,NaN
...,...,...,...,...,...,...,...
480,2026-05-13,1.976000e+06,7126921,1391600.0,1,-0.500392,4.720632e+13
481,2026-05-14,1.970000e+06,6040068,1434950.0,1,0.573532,4.720632e+13
482,2026-05-15,1.819000e+06,7485233,1469100.0,1,0.027851,3.761028e+13
483,2026-05-18,1.840000e+06,6481608,1503350.0,1,0.277661,3.761028e+13


In [26]:
df = pd.merge_asof(df, dart_df, left_index=True, right_on='Date', direction='backward')

In [27]:
df.head(1)

,Close,Volume,MA20,target,NLP_Sentiment,Date,Operation_Profit
Date,,,,,,,
2024-05-20,187236.421875,3592128,NaN,0,-0.242527,2024-05-15,2886029000000


In [28]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

In [29]:
x = df[ ['Volume', 'MA20', 'NLP_Sentiment', 'Operation_Profit'] ]
y = df['target'] 

In [31]:
# 시계열 데이터 변환 -> 앞의 80%를 Train으로 뒤의 20%는 Test
split_idx = int(len(df) * 0.8)
X_train, X_test = x.iloc[:split_idx] , x.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

In [32]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [33]:
pred = model.predict(X_test)

In [34]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.17      1.00      0.29        16
           1       1.00      0.04      0.07        81

    accuracy                           0.20        97
   macro avg       0.59      0.52      0.18        97
weighted avg       0.86      0.20      0.11        97



In [36]:
# 클래스 가중치 부여 
model = RandomForestClassifier(random_state=42, class_weight="balanced_subsample")

model.fit(X_train, y_train)

pred = model.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.17      1.00      0.29        16
           1       1.00      0.04      0.07        81

    accuracy                           0.20        97
   macro avg       0.59      0.52      0.18        97
weighted avg       0.86      0.20      0.11        97



In [ ]:
# 원본 데이터에서 전일 대비 수익율 / 거래량 변화량(%) -> pct_change() 
df['Return'] = df['Close'].pct_change()
df['Volume_change'] = df['Volume'].pct_change()
df.head(3)

In [39]:
# 이동평균선은 절대 수치 대신에 '이격도'로 변환
# 이격도 : 현재 주가가 20일 이평선으로 부터 위/아래로 몇 %나 떨어져있는가?
df['Dist_MA20'] = ( df['Close'] - df['MA20'] ) / df['MA20']

df.dropna(inplace=True)
df.head()

,Close,Volume,MA20,target,NLP_Sentiment,Date,Operation_Profit,Return,Volume_change,Dist_MA20
Date,,,,,,,,,,
2024-06-17,219640.843750,2198340,199942.117188,1,-0.243181,2024-05-15,2886029000000,0.009050,-0.336094,0.098522
2024-06-18,230967.593750,3136231,202128.675781,1,0.547580,2024-05-15,2886029000000,0.051569,0.426636,0.142676
2024-06-19,229982.687500,3758652,204172.418750,1,-0.180261,2024-05-15,2886029000000,-0.004264,0.198461,0.126414
2024-06-20,233922.421875,2927358,206132.442188,1,-0.350277,2024-05-15,2886029000000,0.017131,-0.221168,0.134816
2024-06-21,230475.171875,3848394,207806.835156,1,0.416974,2024-05-15,2886029000000,-0.014737,0.314630,0.109084


In [40]:
x = df[ ['NLP_Sentiment', 'Operation_Profit', 'Return', 'Volume_change', 'Dist_MA20'] ]
y = df['target']

split_idx = int(len(df) * 0.8)
X_train, X_test = x.iloc[:split_idx], x.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

model_final = RandomForestClassifier(random_state=42, class_weight='balanced_subsample')
model_final.fit(X_train, y_train)

pred = model_final.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.67      0.62      0.65        16
           1       0.92      0.94      0.93        78

    accuracy                           0.88        94
   macro avg       0.80      0.78      0.79        94
weighted avg       0.88      0.88      0.88        94

